# TabM

Проверяю табличную DL-модель с параметрически-эффективным ансамблированием на тех же наборах признаков и той же валидационной части.

In [ ]:
from pathlib import Path

import gc
import os
import random
import pickle
import warnings

for warn in [UserWarning, FutureWarning]:
    warnings.filterwarnings("ignore", category=warn)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import optuna

from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import roc_auc_score  # классы несбалансированы
from sklearn.metrics import average_precision_score  # положительный класс -- редкий
from sklearn.metrics import balanced_accuracy_score  # accuracy для дисбаланса
from sklearn.metrics import f1_score
from sklearn.metrics import precision_recall_curve
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", DEVICE)

PROJECT_DIR = Path("/Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton")
DATA_DIR = PROJECT_DIR / "data"
BASE_DIR = DATA_DIR

LOGS_DIR = PROJECT_DIR / "logs" / "tabm_optuna_log"
MODELS_DIR = PROJECT_DIR / "models" / "tabm_models"
SUBMISSIONS_DIR = PROJECT_DIR / "submissions"

LOGS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("BASE_DIR:", BASE_DIR)
print("LOGS_DIR:", LOGS_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("SUBMISSIONS_DIR:", SUBMISSIONS_DIR)

optuna.logging.set_verbosity(optuna.logging.WARNING)

## Загрузка данных

In [ ]:
feature_sets = [
    "top300",
    "top500_clean",
    "top500_lgb_clean",
    "top500_magic_meta",
    "top500_micro_engineered",
]


def resolve_feature_path(folder, split):
    file_name = f"X_{split}_{folder}.parquet"
    candidates = [
        BASE_DIR / folder / file_name,
        BASE_DIR / file_name,
        BASE_DIR / "processed_data" / folder / file_name,
        BASE_DIR / "processed_data" / file_name,
    ]

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        "Не нашел файл с признаками. Проверенные пути:\n"
        + "\n".join(str(path) for path in candidates)
    )


def resolve_target_path(name):
    candidates = [
        BASE_DIR / f"y_{name}.parquet",
        BASE_DIR / "processed_data" / f"y_{name}.parquet",
    ]

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        "Не нашел target-файл. Проверенные пути:\n"
        + "\n".join(str(path) for path in candidates)
    )


def read_feature_matrix(folder, split):
    path = resolve_feature_path(folder, split)
    print(f"read {split:<13} {folder:<25} -> {path}")
    return pd.read_parquet(path)


def read_target(name):
    path = resolve_target_path(name)
    print(f"read target {name:<13} -> {path}")
    return pd.read_parquet(path).iloc[:, 0].values.ravel()


def check_feature_sets():
    rows = []
    for folder in feature_sets:
        row = {"feature_set": folder}
        for split in ["train_sampled", "train_full", "val", "test"]:
            try:
                row[split] = resolve_feature_path(folder, split).exists()
            except FileNotFoundError:
                row[split] = False
        rows.append(row)

    return pd.DataFrame(rows)

check_feature_sets()

## Метрики и подбор порога

In [ ]:
def count_metrics(y_true, y_pred, y_score):
    """
    roc_auc_score: вероятность того, что модель присвоит случайному объекту класса 1 более высокий score, чем случайному объекту класса 0; чем ближе к 1, тем лучше разделение классов, 0.5 — уровень случайного угадывания.

    average_precision_score: среднее качество поиска объектов класса 1 по всем возможным порогам; высокая метрика означает, что среди объектов, которые модель считает наиболее похожими на класс 1, действительно много единиц.

    balanced_accuracy_score: средняя точность отдельно по классу 0 и по классу 1; полезна при дисбалансе классов, потому что не дает модели выглядеть хорошей только за счет угадывания самого частого класса.

    f1_score: одна итоговая оценка качества предсказания класса 1, которая будет высокой только тогда, когда модель одновременно находит много настоящих единиц и не слишком часто ошибочно объявляет нули единицами.
    """

    print("roc auc:", roc_auc_score(y_true, y_score))
    print("average precision:", average_precision_score(y_true, y_score))
    print("balanced accuracy:", balanced_accuracy_score(y_true, y_pred))
    print("f1:", f1_score(y_true, y_pred, zero_division=0))


def best_f1_threshold(y_true, y_score):
    """Подбираем порог по train-score, чтобы .predict() с 0.5 не превращал всё в нули.

    ROC AUC при этом не меняется: он считается по y_score, а не по бинарному y_pred.
    """

    precision, recall, thresholds = precision_recall_curve(y_true, y_score)

    precision = precision[:-1]
    recall = recall[:-1]

    f1 = 2 * precision * recall / (precision + recall + 1e-12)

    if len(thresholds) == 0:
        return 0.5

    return thresholds[np.argmax(f1)]


def predict_with_threshold(y_score, threshold):
    return (y_score >= threshold).astype(int)

## Подготовка матриц

In [ ]:
def prepare_matrices(X_train, X_val):
    scaler = StandardScaler()

    X_train_np = X_train.to_numpy(dtype=np.float32, copy=True)
    X_val_np = X_val.to_numpy(dtype=np.float32, copy=True)

    X_train_np = np.nan_to_num(X_train_np, nan=0.0, posinf=0.0, neginf=0.0)
    X_val_np = np.nan_to_num(X_val_np, nan=0.0, posinf=0.0, neginf=0.0)

    X_train_np = scaler.fit_transform(X_train_np).astype(np.float32)
    X_val_np = scaler.transform(X_val_np).astype(np.float32)

    return X_train_np, X_val_np, scaler


def count_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)

## Архитектура модели

In [ ]:
MODEL_NAME = "TabM"
MODEL_PREFIX = "tabm"


def make_loader(X, y, batch_size, shuffle, params):
    x_tensor = torch.from_numpy(X.astype(np.float32))
    y_tensor = torch.from_numpy(y.astype(np.float32)).view(-1, 1)
    dataset = TensorDataset(x_tensor, y_tensor)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=shuffle,
    )


def make_loader_for_prediction(X, batch_size, params):
    x_tensor = torch.from_numpy(X.astype(np.float32))
    y_tensor = torch.zeros((len(X), 1), dtype=torch.float32)
    dataset = TensorDataset(x_tensor, y_tensor)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )


class BatchEnsembleLinear(nn.Module):
    def __init__(self, in_features, out_features, ensemble_size, bias=True):
        super(BatchEnsembleLinear, self).__init__()

        self.linear = nn.Linear(in_features, out_features, bias=bias)
        self.r = nn.Parameter(torch.randn(ensemble_size, in_features) * 0.02 + 1.0)
        self.s = nn.Parameter(torch.randn(ensemble_size, out_features) * 0.02 + 1.0)

    def forward(self, x):
        # x: batch, ensemble, features
        x = x * self.r.unsqueeze(0)
        x = self.linear(x)
        x = x * self.s.unsqueeze(0)
        return x


class TabMBlock(nn.Module):
    def __init__(self, in_features, out_features, ensemble_size, dropout):
        super(TabMBlock, self).__init__()

        self.linear = BatchEnsembleLinear(in_features, out_features, ensemble_size)
        self.norm = nn.LayerNorm(out_features)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.linear(x)
        x = self.norm(x)
        x = self.activation(x)
        x = self.dropout(x)
        return x


class TabM(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, n_layers=2, ensemble_size=8, dropout=0.2):
        super(TabM, self).__init__()

        self.ensemble_size = ensemble_size
        layers = []
        current_dim = input_dim

        for _ in range(n_layers):
            layers.append(TabMBlock(current_dim, hidden_dim, ensemble_size, dropout))
            current_dim = hidden_dim

        self.backbone = nn.Sequential(*layers)
        self.head = BatchEnsembleLinear(current_dim, 1, ensemble_size)

    def forward(self, x):
        x = x.unsqueeze(1).expand(-1, self.ensemble_size, -1)
        x = self.backbone(x)
        logits = self.head(x).squeeze(-1)
        return logits.mean(dim=1, keepdim=True)


def make_model(input_dim, params):
    return TabM(
        input_dim=input_dim,
        hidden_dim=params["hidden_dim"],
        n_layers=params["n_layers"],
        ensemble_size=params["ensemble_size"],
        dropout=params["dropout"],
    ).to(DEVICE)

## Обучение модели

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train(train)

    total_loss = 0.0
    all_scores = []
    all_targets = []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            if train:
                optimizer.zero_grad(set_to_none=True)

            logits = model(x_batch)
            loss = criterion(logits, y_batch)

            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * len(x_batch)

            scores = torch.sigmoid(logits).detach().cpu().numpy().ravel()
            targets = y_batch.detach().cpu().numpy().ravel()
            all_scores.append(scores)
            all_targets.append(targets)

    all_scores = np.concatenate(all_scores)
    all_targets = np.concatenate(all_targets)
    auc = roc_auc_score(all_targets, all_scores)
    avg_loss = total_loss / len(loader.dataset)

    return avg_loss, auc


def train_model(model, train_loader, val_loader, criterion, optimizer, max_epochs=10, patience=2):
    best_auc = -np.inf
    best_state = None
    history = []
    patience_counter = 0

    for epoch in range(1, max_epochs + 1):
        train_loss, train_auc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_auc = run_epoch(model, val_loader, criterion)

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_auc": train_auc,
                "val_loss": val_loss,
                "val_auc": val_auc,
            }
        )

        print(
            f"epoch {epoch:02d}/{max_epochs} | "
            f"train loss {train_loss:.5f} auc {train_auc:.5f} | "
            f"val loss {val_loss:.5f} auc {val_auc:.5f}"
        )

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return best_auc, history, best_state


def load_checkpoint(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


@torch.no_grad()
def predict_scores(model, loader):
    model.eval()
    all_scores = []

    for x_batch, _ in loader:
        x_batch = x_batch.to(DEVICE)
        logits = model(x_batch)
        scores = torch.sigmoid(logits).detach().cpu().numpy().ravel()
        all_scores.append(scores)

    return np.concatenate(all_scores)

## Функция для оптимизации параметров с помощью optuna

In [ ]:
def objective(trial, X_train, y_train, X_val, y_val, max_epochs, patience):
    params = {
        "hidden_dim": trial.suggest_categorical("hidden_dim", [128, 256, 384]),
        "n_layers": trial.suggest_int("n_layers", 1, 3),
        "ensemble_size": trial.suggest_categorical("ensemble_size", [4, 8, 12]),
        "dropout": trial.suggest_float("dropout", 0.05, 0.35),
        "lr": trial.suggest_float("lr", 1e-4, 2e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [512, 1024, 2048, 4096]),
        "pos_weight_mult": trial.suggest_float("pos_weight_mult", 0.5, 1.5),
    }

    train_loader = make_loader(X_train, y_train, params["batch_size"], shuffle=True, params=params)
    val_loader = make_loader(X_val, y_val, params["batch_size"], shuffle=False, params=params)

    model = make_model(X_train.shape[1], params)

    pos_count = np.maximum(y_train.sum(), 1)
    neg_count = len(y_train) - y_train.sum()
    pos_weight = (neg_count / pos_count) * params["pos_weight_mult"]
    pos_weight = torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=params["weight_decay"],
    )

    best_auc, history, _ = train_model(
        model,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        max_epochs=max_epochs,
        patience=patience,
    )

    trial.set_user_attr("n_params", count_parameters(model))
    trial.set_user_attr("last_epoch", history[-1]["epoch"])

    del model, train_loader, val_loader, criterion, optimizer
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    elif DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return best_auc

In [ ]:
def logging_callback(study, trial):
    set_name = study.user_attrs.get("set_name", "unknown")
    train_mode = study.user_attrs.get("train_mode", "unknown")
    log_file_path = LOGS_DIR / f"{set_name}_{train_mode}_optimization.log"

    with open(log_file_path, "a") as f:
        f.write(
            f"Trial {trial.number:03d} finished | "
            f"ROC-AUC: {trial.value:.5f} | "
            f"Best ROC-AUC so far: {study.best_value:.5f}\n"
        )

## Подбор параметров на усеченной обучающей выборке

In [ ]:
OPTUNA_N_TRIALS_SAMPLED = 20
OPTUNA_TIMEOUT_SAMPLED = 1800
SAMPLED_MAX_EPOCHS = 6
SAMPLED_PATIENCE = 2

y_train_sampled = read_target("train_sampled")
y_val = read_target("val")

best_params_sampled = {}
best_scores_sampled = {}

In [ ]:
for folder in feature_sets:
    print(f"\n==================================================")
    print(f" НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: {folder}")
    print(f"==================================================")

    X_train_sampled = read_feature_matrix(folder, "train_sampled")
    X_val = read_feature_matrix(folder, "val")

    assert len(X_train_sampled) == len(y_train_sampled)
    assert len(X_val) == len(y_val)

    X_train_sampled_np, X_val_np, _ = prepare_matrices(X_train_sampled, X_val)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.set_user_attr("set_name", folder)
    study.set_user_attr("train_mode", "sampled")

    log_file_path = LOGS_DIR / f"{folder}_sampled_optimization.log"
    if log_file_path.exists():
        log_file_path.unlink()

    study.optimize(
        lambda trial: objective(
            trial,
            X_train_sampled_np,
            y_train_sampled,
            X_val_np,
            y_val,
            max_epochs=SAMPLED_MAX_EPOCHS,
            patience=SAMPLED_PATIENCE,
        ),
        n_trials=OPTUNA_N_TRIALS_SAMPLED,
        timeout=OPTUNA_TIMEOUT_SAMPLED,
        callbacks=[logging_callback],
        show_progress_bar=True,
    )

    best_params_sampled[folder] = study.best_params
    best_scores_sampled[folder] = study.best_value

    print(f"\n УСПЕШНО ЗАВЕРШЕНО ДЛЯ: {folder}")
    print(f" Лучший ROC-AUC на валидации: {study.best_value:.5f}")
    print(f" Подробный лог сохранен в: {log_file_path}")
    print(" Лучшие параметры:")
    for param, value in study.best_params.items():
        print(f"   -> {param}: {value}")

    del X_train_sampled, X_val, X_train_sampled_np, X_val_np, study
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    elif DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print("\n\n==================================================")
print(" ОПТИМИЗАЦИЯ НА УСЕЧЕННЫХ ВЫБОРКАХ ЗАВЕРШЕНА!")
print("==================================================")

In [ ]:
sampled_results_df = pd.DataFrame(
    [
        {
            "feature_set": folder,
            "best_roc_auc": best_scores_sampled[folder],
            "best_params": best_params_sampled[folder],
        }
        for folder in feature_sets
    ]
).sort_values("best_roc_auc", ascending=False).reset_index(drop=True)

sampled_results_df

## Подбор параметров на полной обучающей выборке

In [ ]:
OPTUNA_N_TRIALS_FULL = 20
OPTUNA_TIMEOUT_FULL = 1800
FULL_MAX_EPOCHS = 8
FULL_PATIENCE = 2

y_train_full = read_target("train_full")
y_val = read_target("val")

best_params_per_set = {}
best_scores_per_set = {}

In [ ]:
for folder in feature_sets:
    print(f"\n==================================================")
    print(f" НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: {folder}")
    print(f"==================================================")

    X_train_full = read_feature_matrix(folder, "train_full")
    X_val = read_feature_matrix(folder, "val")

    assert len(X_train_full) == len(y_train_full)
    assert len(X_val) == len(y_val)

    X_train_full_np, X_val_np, _ = prepare_matrices(X_train_full, X_val)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.set_user_attr("set_name", folder)
    study.set_user_attr("train_mode", "full")

    log_file_path = LOGS_DIR / f"{folder}_full_optimization.log"
    if log_file_path.exists():
        log_file_path.unlink()

    study.optimize(
        lambda trial: objective(
            trial,
            X_train_full_np,
            y_train_full,
            X_val_np,
            y_val,
            max_epochs=FULL_MAX_EPOCHS,
            patience=FULL_PATIENCE,
        ),
        n_trials=OPTUNA_N_TRIALS_FULL,
        timeout=OPTUNA_TIMEOUT_FULL,
        callbacks=[logging_callback],
        show_progress_bar=True,
    )

    best_params_per_set[folder] = study.best_params
    best_scores_per_set[folder] = study.best_value

    print(f"\n УСПЕШНО ЗАВЕРШЕНО ДЛЯ: {folder}")
    print(f" Лучший ROC-AUC на валидации: {study.best_value:.5f}")
    print(f" Подробный лог сохранен в: {log_file_path}")
    print(" Лучшие параметры:")
    for param, value in study.best_params.items():
        print(f"   -> {param}: {value}")

    del X_train_full, X_val, X_train_full_np, X_val_np, study
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    elif DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print("\n\n==================================================")
print(" ОПТИМИЗАЦИЯ НА ПОЛНОМ TRAIN ЗАВЕРШЕНА!")
print("==================================================")

In [ ]:
full_results_df = pd.DataFrame(
    [
        {
            "feature_set": folder,
            "best_roc_auc": best_scores_per_set[folder],
            "best_params": best_params_per_set[folder],
        }
        for folder in feature_sets
    ]
).sort_values("best_roc_auc", ascending=False).reset_index(drop=True)

full_results_df

## Финальное обучение моделей с лучшими параметрами на полной обучающей выборке

In [ ]:
FINAL_MAX_EPOCHS = 12
FINAL_PATIENCE = 3

final_val_scores = {}
final_thresholds = {}
saved_model_paths = {}
final_metrics_rows = []

In [ ]:
for folder, params in best_params_per_set.items():
    print(f"\n==================================================")
    print(f" ФИНАЛЬНОЕ ОБУЧЕНИЕ НА ПОЛНОМ ТРЕЙНЕ: {folder}")
    print(f"==================================================")

    X_train_full = read_feature_matrix(folder, "train_full")
    X_val = read_feature_matrix(folder, "val")

    assert len(X_train_full) == len(y_train_full)
    assert len(X_val) == len(y_val)

    X_train_full_np, X_val_np, scaler = prepare_matrices(X_train_full, X_val)

    train_loader = make_loader(X_train_full_np, y_train_full, params["batch_size"], shuffle=True, params=params)
    train_eval_loader = make_loader(X_train_full_np, y_train_full, params["batch_size"], shuffle=False, params=params)
    val_loader = make_loader(X_val_np, y_val, params["batch_size"], shuffle=False, params=params)

    final_model = make_model(X_train_full_np.shape[1], params)
    print("params:", f"{count_parameters(final_model):,}")

    pos_count = np.maximum(y_train_full.sum(), 1)
    neg_count = len(y_train_full) - y_train_full.sum()
    pos_weight = (neg_count / pos_count) * params["pos_weight_mult"]
    pos_weight = torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.AdamW(
        final_model.parameters(),
        lr=params["lr"],
        weight_decay=params["weight_decay"],
    )

    current_score, history, best_state = train_model(
        final_model,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        max_epochs=FINAL_MAX_EPOCHS,
        patience=FINAL_PATIENCE,
    )

    y_score_train = predict_scores(final_model, train_eval_loader)
    threshold = best_f1_threshold(y_train_full, y_score_train)

    y_score_val = predict_scores(final_model, val_loader)
    y_pred_val = predict_with_threshold(y_score_val, threshold)

    final_val_scores[folder] = current_score
    final_thresholds[folder] = threshold

    print(f"{MODEL_NAME} threshold: {threshold}")
    count_metrics(y_val, y_pred_val, y_score_val)

    model_filename = MODELS_DIR / f"{MODEL_PREFIX}_{folder}_{current_score:.5f}.pt"
    torch.save(
        {
            "state_dict": best_state,
            "params": params,
            "input_dim": X_train_full_np.shape[1],
            "feature_set": folder,
            "threshold": threshold,
            "scaler": scaler,
            "history": history,
            "val_roc_auc": current_score,
        },
        model_filename,
    )

    saved_model_paths[folder] = model_filename
    print(f"Модель успешно сохранена: {model_filename}")

    final_metrics_rows.append(
        {
            "feature_set": folder,
            "roc_auc": roc_auc_score(y_val, y_score_val),
            "average_precision": average_precision_score(y_val, y_score_val),
            "balanced_accuracy": balanced_accuracy_score(y_val, y_pred_val),
            "f1": f1_score(y_val, y_pred_val, zero_division=0),
            "threshold": threshold,
            "model_path": str(model_filename),
        }
    )

    del X_train_full, X_val, X_train_full_np, X_val_np, train_loader, train_eval_loader, val_loader
    del final_model, criterion, optimizer, y_score_train, y_score_val, y_pred_val
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    elif DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print("\n\n==================================================")
print(" ОБУЧЕНИЕ И СОХРАНЕНИЕ ВСЕХ МОДЕЛЕЙ ЗАВЕРШЕНЫ!")
print("==================================================")

## Сравнение финальных моделей

In [ ]:
final_results_df = pd.DataFrame(final_metrics_rows).sort_values(
    "roc_auc", ascending=False
).reset_index(drop=True)

final_results_df.style.format(
    {
        "roc_auc": "{:.6f}",
        "average_precision": "{:.6f}",
        "balanced_accuracy": "{:.6f}",
        "f1": "{:.6f}",
        "threshold": "{:.6f}",
    }
)

In [ ]:
metrics_to_plot = ["roc_auc", "average_precision", "balanced_accuracy", "f1"]

final_results_df.set_index("feature_set")[metrics_to_plot].plot(
    kind="bar", figsize=(10, 5), rot=30, title=f"Сравнение {MODEL_NAME} по наборам признаков"
)

plt.ylabel("score")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
best_folder = final_results_df.loc[0, "feature_set"]
best_model_path = Path(final_results_df.loc[0, "model_path"])

print("best feature set:", best_folder)
print("best model:", best_model_path)
print("best val roc auc:", final_results_df.loc[0, "roc_auc"])
print("best threshold:", final_results_df.loc[0, "threshold"])

## Confusion matrix для лучшей модели

In [ ]:
checkpoint = load_checkpoint(best_model_path)
best_params = checkpoint["params"]
scaler = checkpoint["scaler"]

X_val_best = read_feature_matrix(best_folder, "val")
X_val_best_np = X_val_best.to_numpy(dtype=np.float32, copy=True)
X_val_best_np = np.nan_to_num(X_val_best_np, nan=0.0, posinf=0.0, neginf=0.0)
X_val_best_np = scaler.transform(X_val_best_np).astype(np.float32)

val_loader_best = make_loader(X_val_best_np, y_val, best_params["batch_size"], shuffle=False, params=best_params)

best_model = make_model(X_val_best_np.shape[1], best_params).to(DEVICE)
best_model.load_state_dict(checkpoint["state_dict"])

y_score_best = predict_scores(best_model, val_loader_best)
y_pred_best = predict_with_threshold(
    y_score_best, final_thresholds.get(best_folder, final_results_df.loc[0, "threshold"])
)

ConfusionMatrixDisplay.from_predictions(y_val, y_pred_best)
plt.title(f"{MODEL_NAME}: {best_folder}")
plt.show()

count_metrics(y_val, y_pred_best, y_score_best)

del X_val_best, X_val_best_np, val_loader_best, best_model, y_score_best, y_pred_best
gc.collect()
if DEVICE.type == "mps":
    torch.mps.empty_cache()
elif DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## Предсказание для test и сабмит

In [ ]:
RUN_SUBMISSION = True

if RUN_SUBMISSION:
    checkpoint = load_checkpoint(best_model_path)
    best_params = checkpoint["params"]
    scaler = checkpoint["scaler"]

    X_test = read_feature_matrix(best_folder, "test")
    X_test_np = X_test.to_numpy(dtype=np.float32, copy=True)
    X_test_np = np.nan_to_num(X_test_np, nan=0.0, posinf=0.0, neginf=0.0)
    X_test_np = scaler.transform(X_test_np).astype(np.float32)

    test_loader = make_loader_for_prediction(X_test_np, best_params["batch_size"], best_params)

    best_model = make_model(X_test_np.shape[1], best_params).to(DEVICE)
    best_model.load_state_dict(checkpoint["state_dict"])
    test_score = predict_scores(best_model, test_loader)

    submit_candidates = [
        DATA_DIR / "sample_submission.csv",
        DATA_DIR / "submit.csv",
        PROJECT_DIR / "sample_submission.csv",
        PROJECT_DIR / "submit.csv",
    ]
    submit_path = next((path for path in submit_candidates if path.exists()), None)

    if submit_path is not None:
        submission = pd.read_csv(submit_path)
        target_col = "target" if "target" in submission.columns else submission.columns[-1]
        submission[target_col] = test_score
    else:
        submission = pd.DataFrame(
            {
                "index": X_test.index,
                "target": test_score,
            }
        )

    submission_path = SUBMISSIONS_DIR / f"{MODEL_PREFIX}_{best_folder}_{final_results_df.loc[0, 'roc_auc']:.5f}.csv"
    submission.to_csv(submission_path, index=False)

    print("submission saved:", submission_path)
    display(submission.head())

    del X_test, X_test_np, test_loader, best_model, test_score
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    elif DEVICE.type == "cuda":
        torch.cuda.empty_cache()